# CAMUS dataset explorer

Loads `CAMUSDataset` and plots samples with their expert segmentations.

Labels: `1` LV endocardium (cavity) · `2` LV myocardium · `3` left atrium.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.colors import to_rgb
from matplotlib.lines import Line2D

from camus_dataset import LABELS, NUM_CLASSES, CAMUSDataset

DATA_ROOT = Path.cwd().parent / "CAMUS_public"
assert DATA_ROOT.is_dir(), f"CAMUS_public not found at {DATA_ROOT} - fix DATA_ROOT"

plt.rcParams["figure.dpi"] = 110
print(f"data root: {DATA_ROOT}")

## 1. Initialise the dataset

In [ ]:
train_ds = CAMUSDataset(DATA_ROOT, split="train")
val_ds = CAMUSDataset(DATA_ROOT, split="val")
test_ds = CAMUSDataset(DATA_ROOT, split="test")

for ds in (train_ds, val_ds, test_ds):
    print(f"{ds.split:<6s} {len(ds.patients):>3d} patients  {len(ds):>4d} samples")

# The official splits are patient-level, so no patient appears in two of them.
patient_sets = [{s.patient for s in ds.samples} for ds in (train_ds, val_ds, test_ds)]
assert not (patient_sets[0] & patient_sets[1] | patient_sets[0] & patient_sets[2] | patient_sets[1] & patient_sets[2])
print("\nno patient overlap between splits")

## 2. Inspect a single sample

In [ ]:
sample = train_ds[0]

print(sample["key"])
for field in ("view", "instant", "image_quality", "ef", "sex", "age"):
    print(f"  {field:<14s} {sample[field]}")
print(f"  image          {tuple(sample['image'].shape)} {sample['image'].dtype} "
      f"range [{sample['image'].min():.2f}, {sample['image'].max():.2f}]")
print(f"  label          {tuple(sample['label'].shape)} {sample['label'].dtype} "
      f"classes {sorted(sample['label'].unique().tolist())}")
print(f"  native size    {tuple(sample['original_size'].tolist())}")
print(f"  spacing        {[round(s, 4) for s in sample['spacing'].tolist()]} mm/px (row, col, after resize)")

## 3. Plotting helpers

`spacing` is carried per sample and corrected for the resize, so pixel counts can
be converted back to mm² without going near the original file.

In [ ]:
CLASS_COLORS = {1: "#e8564a", 2: "#2fb8a0", 3: "#4a7fe8"}


def _to_numpy(sample):
    return sample["image"][0].numpy(), sample["label"].numpy()


def overlay_labels(ax, label, alpha=0.35, contours=True):
    """Paint each class over the image, plus a crisp contour on the boundary."""
    for cls, color in CLASS_COLORS.items():
        mask = label == cls
        if not mask.any():
            continue
        rgba = np.zeros((*mask.shape, 4))
        rgba[mask] = (*to_rgb(color), alpha)
        ax.imshow(rgba, interpolation="nearest")
        if contours:
            ax.contour(mask.astype(float), levels=[0.5], colors=[color], linewidths=1.1)


def legend_handles():
    return [Line2D([0], [0], color=CLASS_COLORS[c], lw=3, label=f"{c} {LABELS[c]}")
            for c in CLASS_COLORS]


def show_sample(sample, axes=None, title=None):
    """Three panels: B-mode image, label map, and the two superimposed."""
    image, label = _to_numpy(sample)
    if axes is None:
        _, axes = plt.subplots(1, 3, figsize=(11, 4), layout="constrained")

    axes[0].imshow(image, cmap="gray")
    axes[0].set_title("B-mode")

    axes[1].imshow(np.zeros_like(image), cmap="gray", vmin=0, vmax=1)
    overlay_labels(axes[1], label, alpha=1.0, contours=False)
    axes[1].set_title("label")

    axes[2].imshow(image, cmap="gray")
    overlay_labels(axes[2], label)
    axes[2].set_title("overlay")

    for ax in axes:
        ax.set_xticks([]); ax.set_yticks([])
    axes[0].set_ylabel(title or sample["key"], fontsize=9)
    return axes


def find_index(dataset, patient, view, instant):
    """Position of a specific (patient, view, instant) sample in `dataset`."""
    target = f"{patient}_{view}_{instant}"
    for i, key in enumerate(dataset.samples):
        if str(key) == target:
            return i
    raise KeyError(f"{target} is not in this split")

## 4. One sample

In [ ]:
axes = show_sample(train_ds[0])
axes[2].legend(handles=legend_handles(), loc="lower right", fontsize=7, framealpha=0.85)
axes[0].figure.suptitle(
    f"{sample['key']}  ·  quality={sample['image_quality']}  ·  EF={sample['ef']:.0f}%", fontsize=10)
plt.show()

## 5. A grid of random samples

In [ ]:
def show_grid(dataset, n=6, seed=0):
    rng = np.random.default_rng(seed)
    picks = rng.choice(len(dataset), size=n, replace=False)
    ncols = 3
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.1 * ncols, 3.7 * nrows),
                             layout="constrained")

    for ax, idx in zip(axes.flat, picks):
        s = dataset[int(idx)]
        image, label = _to_numpy(s)
        ax.imshow(image, cmap="gray")
        overlay_labels(ax, label)
        ax.set_title(f"{s['patient']} {s['view']} {s['instant']}\n"
                     f"quality={s['image_quality']}  EF={s['ef']:.0f}%", fontsize=8)
    for ax in axes.flat:
        ax.set_xticks([]); ax.set_yticks([])
    for ax in axes.flat[n:]:
        ax.axis("off")

    fig.legend(handles=legend_handles(), loc="outside lower center", ncol=3,
               fontsize=8, frameon=False)
    plt.show()


show_grid(train_ds, n=6)

## 6. ED vs ES for one patient

Both apical views at both instants. The cavity (red) is visibly smaller at ES,
and the atrium (blue) larger — the contraction the EF calculation measures.

In [ ]:
patient = train_ds.patients[0]
fig, axes = plt.subplots(2, 2, figsize=(7.5, 8.4), layout="constrained")

for row, view in enumerate(("2CH", "4CH")):
    for col, instant in enumerate(("ED", "ES")):
        s = train_ds[find_index(train_ds, patient, view, instant)]
        image, label = _to_numpy(s)
        ax = axes[row, col]
        ax.imshow(image, cmap="gray")
        overlay_labels(ax, label)

        mm2 = float(s["spacing"].prod())
        lv_area = float((s["label"] == 1).sum()) * mm2 / 100.0
        ax.set_title(f"{view} {instant}   LV area {lv_area:.1f} cm²", fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])

fig.suptitle(f"{patient}  ·  reported EF {s['ef']:.0f}%", fontsize=11)
fig.legend(handles=legend_handles(), loc="outside lower center", ncol=3,
           fontsize=8, frameon=False)
plt.show()

## 7. Batches from a DataLoader

In [ ]:
loader = torch.utils.data.DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=0)
batch = next(iter(loader))

print(f"image {tuple(batch['image'].shape)} {batch['image'].dtype}")
print(f"label {tuple(batch['label'].shape)} {batch['label'].dtype}")
print(f"patients in batch: {batch['patient']}")

counts = torch.bincount(batch["label"].flatten(), minlength=NUM_CLASSES).float()
for cls, name in LABELS.items():
    print(f"  {cls} {name:<16s} {counts[cls] / counts.sum():6.2%}")